# Industry Grade Project 1 — Reproducible DevOps → DevSecOps Runbook

**Source fidelity:** Reconstructed from the original Word exercise. Original commands, scripts, screenshots, and completion logs are preserved in the source document; this notebook annotates the execution sequence and adds clearly labeled modernization extensions.

**Tags:** `jenkins` `maven` `tomcat` `docker` `ansible` `kubernetes` `prometheus` `devsecops` `java-war`

## Learning path
1. Jenkins + Maven build and Tomcat deployment
2. Docker image build and standalone container deployment
3. Ansible image automation
4. Container registry promotion
5. Kubernetes Deployment → ReplicaSet → 3 Pod replicas
6. LoadBalancer Service validation
7. Prometheus metrics, logging, and alerting
8. Optional enterprise security gates


## 1. Jenkins: source, clean build, test, package, deploy

The original exercise configures Jenkins to pull the repository, run Maven `clean install`, archive the WAR/JAR artifacts, and deploy the WAR to Tomcat. The recorded build evidence shows the Maven reactor completing successfully and Jenkins deploying `webapp.war`.

**Original command:**
```bash
mvn clean install
```

**Completion evidence from the exercise:** Git checkout succeeded; the parent POM, server JAR, and webapp WAR built; server tests passed; Jenkins archived the artifacts; and Tomcat deployment ended with `Finished: SUCCESS`.

Modernization note: preserve the freestyle-job evidence for the portfolio, but prefer Pipeline-as-Code with a `Jenkinsfile` for reproducibility.


## 2. Docker: build and deploy the WAR

The original exercise transfers the WAR to the Docker host and executes this sequence:

```bash
cd /opt/docker
docker build -t regappv1 .
docker stop registerapp
docker rm registerapp
docker run -d --name registerapp -p 8087:8080 regappv1
```

The original Dockerfile is:

```dockerfile
FROM tomcat:latest
RUN cp -R /usr/local/tomcat/webapps.dist/* /usr/local/tomcat/webapps
COPY ./*.war /usr/local/tomcat/webapps
```

The exercise log shows the image named `docker.io/library/regappv1`, a container ID returned by `docker run`, SSH transfer completion, and `Finished: SUCCESS`. The first-run `No such container` messages for stop/remove are expected when the named container does not yet exist. For production, pin the Tomcat version/digest instead of `latest`.


## 3. Ansible: automate image build and registry promotion

The original exercise uses an Ansible controller to build, tag, and push the image.

```yaml
---
- hosts: ansible
  tasks:
    - name: create docker image
      command: docker build -t regapp:latest .
      args:
        chdir: /opt/docker
    - name: create tag to push image onto dockerhub repository
      command: docker tag regapp:latest shotokan777/regapp:latest
    - name: push docker image
      command: docker push shotokan777/regapp:latest
```

Jenkins transfers `webapp.war` to the Ansible server and runs:

```bash
docker login
ansible-playbook /opt/docker/create_image_regapp.yml
sleep 10
```

The original log shows successful build/tag/push tasks. It also records an unreachable inventory entry named `~`; this is useful troubleshooting evidence and should be removed from a cleaned inventory.


## 4. Registry promotion: original Docker Hub path and AWS ECR modernization

The historical lab pushed `shotokan777/regapp:latest`. An AWS-oriented enterprise equivalent is ECR:

```bash
aws ecr get-login-password --region <region> \
  | docker login --username AWS --password-stdin <account>.dkr.ecr.<region>.amazonaws.com

docker tag regapp:latest <account>.dkr.ecr.<region>.amazonaws.com/regapp:<version>
docker push <account>.dkr.ecr.<region>.amazonaws.com/regapp:<version>
```

Use Jenkins credentials binding or workload identity/instance roles. Do not place AWS keys in the notebook, repository, Jenkinsfile, or screenshots.


## 5. Kubernetes: deploy three replicas and expose through a Service

The original exercise uses Ansible to apply the Deployment and Service manifests and restart the Deployment after a new image is published.

```yaml
---
- hosts: kubernetes
  user: root
  tasks:
    - name: deploy regapp on kubernetes
      command: kubectl apply -f regapp-deployment.yml
    - name: create service for regapp
      command: kubectl apply -f regapp-service.yml
    - name: update deployment with new pods if image updated
      command: kubectl rollout restart deployment.apps/valaxy-regapp
```

```mermaid
flowchart LR
    J[Jenkins CI] --> A[Ansible Controller]
    A --> R[Container Registry]
    A --> K[Cluster API]
    K --> D[Deployment]
    D --> RS[ReplicaSet]
    RS --> P1[Pod Replica 1]
    RS --> P2[Pod Replica 2]
    RS --> P3[Pod Replica 3]
    LB[External Load Balancer] --> S[Service]
    S --> P1
    S --> P2
    S --> P3
```

Validation commands:
```bash
kubectl apply -f regapp-deployment.yml
kubectl apply -f regapp-service.yml
kubectl rollout status deployment/valaxy-regapp
kubectl get deployment,rs,pods,svc -o wide
```


## 6. CI → CD trigger and completion evidence

The original runbook separates the CI job from a downstream CD job. CI builds the Maven artifacts and runs the Ansible image workflow; completion triggers the Kubernetes CD job. This is a useful separation-of-duties pattern.

Completion checks:
```bash
kubectl get pods -l app=regapp
kubectl get svc valaxy-service
kubectl describe deployment valaxy-regapp
kubectl rollout history deployment/valaxy-regapp
```

Expected state: desired replicas = 3, available replicas = 3, and the Service has a reachable external endpoint when the cloud load balancer is provisioned.


## 7. Observability extension: Prometheus, logs, metrics, and alerts

**Add-on:** this section does not claim the original Word lab installed Prometheus.

```bash
helm repo add prometheus-community https://prometheus-community.github.io/helm-charts
helm repo update
helm upgrade --install monitoring prometheus-community/kube-prometheus-stack \
  --namespace monitoring --create-namespace
kubectl get pods -n monitoring
kubectl get servicemonitors -A
```

Useful PromQL:
```promql
sum(rate(http_server_requests_seconds_count[5m])) by (status)
histogram_quantile(0.95, sum(rate(http_server_requests_seconds_bucket[5m])) by (le))
sum(rate(container_cpu_usage_seconds_total{pod=~"valaxy-regapp.*"}[5m])) by (pod)
sum(container_memory_working_set_bytes{pod=~"valaxy-regapp.*"}) by (pod)
kube_deployment_status_replicas_unavailable{deployment="valaxy-regapp"}
```

### Mock monitoring dashboard — illustrative, not original lab evidence
```mermaid
flowchart TB
  subgraph Mock_Prometheus_Dashboard
    A[Request rate: 42 req/s]
    B[p95 latency: 180 ms]
    C[5xx rate: 0.3%]
    D[Available replicas: 3/3]
    E[CPU per pod: 32% / 29% / 35%]
    F[Memory per pod: 410 / 398 / 425 MiB]
  end
```


### Alerting rules
```yaml
groups:
- name: regapp-alerts
  rules:
  - alert: RegAppReplicaUnavailable
    expr: kube_deployment_status_replicas_unavailable{deployment="valaxy-regapp"} > 0
    for: 5m
    labels:
      severity: warning
  - alert: RegAppHigh5xxRate
    expr: |
      sum(rate(http_server_requests_seconds_count{status=~"5.."}[5m]))
      /
      sum(rate(http_server_requests_seconds_count[5m])) > 0.05
    for: 10m
    labels:
      severity: critical
```

A production implementation should ship application logs, Tomcat access logs, container stdout/stderr, Kubernetes events, Kubernetes audit logs, and cloud load-balancer logs into a centralized logging/SIEM platform. Prometheus handles metrics; logs should flow through Fluent Bit or OpenTelemetry Collector into the approved analytics/SIEM backend.


## 8. Optional DevSecOps security add-on

These controls are deliberately separated from the original lab evidence.

```mermaid
flowchart LR
  C[Commit] --> V[Veracode SAST]
  V --> N[Nexus IQ SCA Policy]
  N --> X[Contrast IAST]
  X --> B[Build Image]
  B --> A[Aqua Container Scan]
  A --> R[Registry Promotion]
  R --> K[Kubernetes Test]
  K --> U[Burp Suite DAST]
  U --> P[Promote or Block]
```

| Gate | Purpose | Release decision |
|---|---|---|
| Veracode | Static code security analysis | Block policy-breaking findings |
| Nexus IQ | Open-source dependency governance | Block prohibited vulnerable components/licenses |
| Contrast | Runtime/IAST findings during tests | Triage exploitable application paths |
| Aqua | Image, package, secret, and container posture checks | Block unsafe image promotion |
| Burp Suite | DAST against test deployment | Block exploitable web findings |


## 9. Privacy and evidence handling

The notebook preserves the educational sequence while avoiding unnecessary publication of live credentials, private IPs, account IDs, tokens, internal hostnames, or secret values. Historical screenshots remain in the original Word document as exercise evidence; any live secrets should be rotated and sensitive identifiers redacted before public release.

## 10. End-to-end mental model

**Git → Jenkins → Maven → WAR → Docker → Ansible → Registry/ECR → Kubernetes Deployment → ReplicaSet → Pods → Service/LoadBalancer → Prometheus metrics + centralized logs → security gates and release decision.**
